# Virtual Environments (Advanced)

This notebook explains what actually happens under the hood when you activate a virtual environment. Understanding the mechanics makes you a better developer and a better security professional — many real attacks exploit the exact mechanisms described here.

---

## Table of Contents

- [ ] 1. How Activation Works: The PATH
- [ ] 2. Inside the .venv Directory
- [ ] 3. sys.path — Python's Module Search Order
- [ ] 4. Dependency Management in Practice
- [ ] 5. Security Attacks on Package Ecosystems
- [ ] 6. Modern Alternatives: pipenv and poetry
- [ ] 7. Summary


---

## 1. How Activation Works: The PATH

**Analogy:** When your OS needs to run a command, it does not know where every program lives. It searches through a list of folders in order — this list is the **PATH environment variable**. The first match wins. Activation exploits this: it prepends the `.venv/bin` folder to the front of PATH so the venv's Python and pip are found first, before the system ones.

```bash
# Before activation:
PATH = /usr/local/bin:/usr/bin:/bin
#        ^ system Python lives here

# After activation:
PATH = /your/project/.venv/bin:/usr/local/bin:/usr/bin:/bin
#        ^ venv Python is NOW FIRST — wins every time
```

This is the entire secret of virtual environments: **path manipulation**. Nothing magic, just precedence.


In [1]:
# Inspect the current PATH from Python
import os

path_entries = os.environ.get("PATH", "").split(os.pathsep)

print(f"PATH has {len(path_entries)} entries:")
print()
for i, entry in enumerate(path_entries[:10]):
    marker = "  <-- FIRST: wins if multiple python binaries exist" if i == 0 else ""
    print(f"  [{i+1}] {entry}{marker}")

if len(path_entries) > 10:
    print(f"  ... ({len(path_entries) - 10} more entries)")


PATH has 13 entries:

  [1] c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv\Scripts  <-- FIRST: wins if multiple python binaries exist
  [2] C:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv\Scripts
  [3] C:\Users\Zoltan\AppData\Local\Programs\Microsoft VS Code
  [4] C:\Windows\system32
  [5] C:\Windows
  [6] C:\Windows\System32\Wbem
  [7] C:\Windows\System32\WindowsPowerShell\v1.0\
  [8] C:\Windows\System32\OpenSSH\
  [9] C:\Program Files\PowerShell\7\
  [10] C:\Program Files\Git\cmd
  ... (3 more entries)


In [2]:
# Find where the currently active python lives in the PATH
import sys
import os

python_exe   = sys.executable
python_dir   = os.path.dirname(python_exe)
path_entries = os.environ.get("PATH", "").split(os.pathsep)

print("Active Python executable:")
print(" ", python_exe)
print()

# Find its position in PATH
for i, entry in enumerate(path_entries):
    if os.path.normcase(python_dir) == os.path.normcase(entry):
        print(f"Found at PATH position {i+1} (position 1 = highest priority)")
        if i == 0:
            print("[OK] Virtual environment is at the front of PATH")
        else:
            print("[NOTE] Not at position 1 — check your activation")
        break


Active Python executable:
  c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv\Scripts\python.exe

Found at PATH position 1 (position 1 = highest priority)
[OK] Virtual environment is at the front of PATH


### Practice - Section 1

**Write** — Run the cell below with your `.venv` active. It prints your full PATH. Identify which entry was prepended by the activation. How can you tell it is the venv entry?


In [ ]:
# Your code here
# Print each PATH entry on its own line
# Mark any entry that appears to be inside a .venv folder
import os



<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import os

path_entries = os.environ.get("PATH", "").split(os.pathsep)

for entry in path_entries:
    is_venv = ".venv" in entry or "venv" in entry.lower()
    label = "  <-- virtual environment" if is_venv else ""
    print(f"{entry}{label}")
```

</details>

---

## 2. Inside the .venv Directory

When you run `python -m venv .venv`, Python builds a specific directory structure. Understanding it explains why the environment is isolated.

```
.venv/
  |- bin/                 (Linux/macOS) or Scripts/ (Windows)
  |    |- python          symlink to the system Python interpreter
  |    |- python3         same
  |    |- pip             pip wrapper that installs into THIS env only
  |    |- activate        the shell script that modifies PATH
  |
  |- lib/
  |    |- python3.XX/
  |         |- site-packages/    <-- ALL installed packages live here
  |              |- requests/
  |              |- scapy/
  |              |- ...
  |
  |- include/             C header files (for packages with C extensions)
  |
  |- pyvenv.cfg           configuration — points back to the system Python
```

The `python` binary in `.venv/bin/` is **not a full Python installation** — it is a symlink (or small wrapper) that points to the system Python. What is truly isolated is `lib/pythonX.X/site-packages/` — the folder where installed packages live.


In [3]:
# Read the pyvenv.cfg file to see the venv configuration
import sys
from pathlib import Path

in_venv = sys.prefix != sys.base_prefix

if in_venv:
    cfg_path = Path(sys.prefix) / "pyvenv.cfg"
    if cfg_path.exists():
        print("pyvenv.cfg contents:")
        print("-" * 40)
        print(cfg_path.read_text())
    else:
        print("pyvenv.cfg not found at", cfg_path)
else:
    print("Not inside a virtual environment. Activate one first.")


pyvenv.cfg contents:
----------------------------------------
home = C:\Users\Zoltan\AppData\Local\Python\pythoncore-3.14-64
include-system-site-packages = false
version = 3.14.3
executable = C:\Users\Zoltan\AppData\Local\Python\pythoncore-3.14-64\python.exe
command = C:\Users\Zoltan\AppData\Local\Python\pythoncore-3.14-64\python.exe -m venv C:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv



In [4]:
# Find the site-packages folder for the active environment
import site
import os

site_packages = site.getsitepackages()

print("site-packages directories for this environment:")
for path in site_packages:
    print(" ", path)

# Count installed packages in the first site-packages folder
from pathlib import Path

if site_packages:
    sp_path = Path(site_packages[0])
    if sp_path.exists():
        entries = [e for e in sp_path.iterdir() if e.is_dir() and not e.name.startswith("_")]
        print(f"\nPackage directories in site-packages: {len(entries)}")


site-packages directories for this environment:
  c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv
  c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv\Lib\site-packages

Package directories in site-packages: 4


### Practice - Section 2

**Write** — Navigate to your `.venv` folder using Python's `pathlib` and list the contents of the `bin/` (or `Scripts/` on Windows) directory. Identify the `activate` script and the `pip` wrapper.


In [ ]:
# Your code here
import sys
from pathlib import Path



<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import sys
from pathlib import Path

in_venv = sys.prefix != sys.base_prefix
if not in_venv:
    print("Not in a venv. Activate one first.")
else:
    venv_root = Path(sys.prefix)

    # bin on Linux/macOS, Scripts on Windows
    bin_dir = venv_root / "bin" if (venv_root / "bin").exists() else venv_root / "Scripts"

    print(f"Listing: {bin_dir}")
    print("-" * 40)
    for item in sorted(bin_dir.iterdir()):
        highlight = "  <-- activation script" if "activate" in item.name else ""
        highlight = "  <-- pip wrapper" if item.name == "pip" else highlight
        print(f"  {item.name}{highlight}")
```

</details>

---

## 3. sys.path — Python's Module Search Order

**Analogy:** When your script says `import requests`, Python does not know where `requests` lives. It searches through a list of folders in order — this is `sys.path`. The first folder that contains a matching module wins. Virtual environments work by ensuring their `site-packages` folder is near the top of this list.

`sys.path` is the import equivalent of the OS PATH: **order determines priority**.


In [5]:
# Inspect the current module search path
import sys

print(f"sys.path has {len(sys.path)} entries:")
print()
for i, entry in enumerate(sys.path):
    is_venv = ".venv" in entry or "site-packages" in entry
    label = "  <-- packages searched here" if "site-packages" in entry else ""
    print(f"  [{i}] {entry}{label}")


sys.path has 7 entries:

  [0] C:\Users\Zoltan\AppData\Local\Python\pythoncore-3.14-64\python314.zip
  [1] C:\Users\Zoltan\AppData\Local\Python\pythoncore-3.14-64\DLLs
  [2] C:\Users\Zoltan\AppData\Local\Python\pythoncore-3.14-64\Lib
  [3] C:\Users\Zoltan\AppData\Local\Python\pythoncore-3.14-64
  [4] c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv
  [5] 
  [6] c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv\Lib\site-packages  <-- packages searched here


In [6]:
# Trace where a specific module is being imported from
# This confirms you are using the venv version, not a system-level install

try:
    import requests
    print("requests location:")
    print(" ", requests.__file__)
    print()
    # If the path contains .venv, you are using the isolated version
    if ".venv" in requests.__file__:
        print("[OK] Using the virtual environment version")
    else:
        print("[WARNING] Using a system-level or unexpected version")
except ImportError:
    print("requests is not installed. Run: pip install requests")


requests location:
  c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv\Lib\site-packages\requests\__init__.py

[OK] Using the virtual environment version


### Practice - Section 3

**Write** — Without importing any module, write a function `find_module_location(module_name)` that searches `sys.path` manually and prints whether the module folder exists in each path entry. This is what Python does internally on every `import` statement.


In [ ]:
# Your code here
import sys
from pathlib import Path

# Write a function that manually searches sys.path for a module folder
def find_module_location(module_name):
    pass   # replace with your implementation

find_module_location("requests")


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import sys
from pathlib import Path

def find_module_location(module_name):
    """Search sys.path for a module folder, mimicking Python's import mechanism."""
    found = False
    for i, search_path in enumerate(sys.path):
        candidate = Path(search_path) / module_name
        if candidate.exists():
            print(f"  [FOUND at sys.path[{i}]] {candidate}")
            found = True
            break   # Python uses the first match
    if not found:
        print(f"  [NOT FOUND] '{module_name}' not in any sys.path entry")
        print(f"  Install it with: pip install {module_name}")

find_module_location("requests")
find_module_location("scapy")
find_module_location("os")        # built-in module — will be found in stdlib
```

</details>

---

## 4. Dependency Management in Practice

Real projects grow complex. You will encounter dependency conflicts, version pinning decisions, and the need to separate development-only tools from production requirements.

### Pinned vs unpinned requirements

```
# Pinned (production) — exact versions, fully reproducible
requests==2.31.0
scapy==2.5.0

# Unpinned (flexible) — latest compatible version
requests>=2.28
scapy
```

**Pinned:** Safer for production. Every team member runs exactly the same code.

**Unpinned:** Easier to maintain but risks "it worked yesterday" surprises after an upstream update.

### Separating dev from production requirements

A common pattern is to have two files:

```
requirements.txt          -- production only (what runs in deployment)
requirements-dev.txt      -- production + testing + linting tools
```

`requirements-dev.txt` typically starts with:
```
-r requirements.txt       # include everything from the base file
pytest==8.1.0
black==24.3.0
```


In [7]:
# Validate that all packages in a requirements.txt are installed
# A useful pre-flight check before running a security script

import importlib.metadata
from packaging.version import Version, InvalidVersion   # needs: pip install packaging

requirements = {
    "requests":  "2.28.0",
    "scapy":     "2.5.0",
    "paramiko":  "3.0.0",
}

all_ok = True
for package, min_version in requirements.items():
    try:
        installed = importlib.metadata.version(package)
        if Version(installed) >= Version(min_version):
            print(f"[OK]      {package:<15} {installed} (>= {min_version} required)")
        else:
            print(f"[OLD]     {package:<15} {installed} -- need >= {min_version}")
            all_ok = False
    except importlib.metadata.PackageNotFoundError:
        print(f"[MISSING] {package:<15} not installed")
        all_ok = False

print()
print("[PASS] All dependencies met" if all_ok else "[FAIL] Missing or outdated dependencies")


[OK]      requests        2.32.5 (>= 2.28.0 required)
[MISSING] scapy           not installed
[MISSING] paramiko        not installed

[FAIL] Missing or outdated dependencies


### Practice - Section 4

**Write** — Build a function `validate_requirements(req_file_path)` that reads a `requirements.txt` file (pinned format: `name==version`) and checks whether each listed package is installed at that exact version. Print `[OK]`, `[MISMATCH]`, or `[MISSING]` for each line.


In [ ]:
# Your code here
import importlib.metadata
from pathlib import Path

def validate_requirements(req_file_path):
    pass   # replace with your implementation


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import importlib.metadata
from pathlib import Path

def validate_requirements(req_file_path):
    """Check each pinned requirement against what is actually installed."""
    path = Path(req_file_path)
    if not path.exists():
        print(f"File not found: {path}")
        return

    all_ok = True
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or line.startswith("-"):
            continue
        if "==" not in line:
            print(f"[SKIP]     {line} (unpinned — skipping)")
            continue

        name, required_ver = line.split("==")
        name = name.strip()
        required_ver = required_ver.strip()

        try:
            installed_ver = importlib.metadata.version(name)
            if installed_ver == required_ver:
                print(f"[OK]       {name:<20} {installed_ver}")
            else:
                print(f"[MISMATCH] {name:<20} installed: {installed_ver} | required: {required_ver}")
                all_ok = False
        except importlib.metadata.PackageNotFoundError:
            print(f"[MISSING]  {name:<20} not installed")
            all_ok = False

    print()
    print("Result:", "PASS" if all_ok else "FAIL")

# Test with an example file
# validate_requirements("requirements.txt")
print("Function defined. Call: validate_requirements('requirements.txt')")
```

</details>

---

## 5. Security Attacks on Package Ecosystems

Understanding how Python resolves packages (PATH, sys.path, PyPI search) immediately reveals several attack vectors. These are real attacks that have caused incidents at large organisations.

### Attack 1: Typosquatting

An attacker publishes a package on PyPI with a name close to a legitimate popular package, hoping a developer will mistype it:

| Legitimate | Malicious typosquat |
|------------|---------------------|
| `requests` | `requets`, `request`, `requestss` |
| `scapy` | `scappy`, `scapy2` |
| `paramiko` | `paramyko`, `paramiko2` |

The malicious package installs a backdoor, steals credentials, or adds the machine to a botnet — all while appearing to function normally.

**Defence:** Always double-check package names. Verify the author and download count on [pypi.org](https://pypi.org) before installing anything unfamiliar.

### Attack 2: Dependency Confusion

If an organisation uses a private package registry (e.g., a corporate Nexus or Artifactory server) with internal packages named things like `acme-auth-utils`, an attacker can publish a public PyPI package with the **same name but a higher version number**. Depending on the pip configuration, pip may fetch the public package instead of the internal one.

This attack was demonstrated at scale by security researcher Alex Birsan in 2021 — he achieved remote code execution inside Apple, Microsoft, and 33 other major companies using this technique.

**Defence:** Use a private registry that explicitly pins internal package names, or configure pip to only fetch from the internal registry for known internal packages.

### Attack 3: PATH Hijacking

Because the OS searches PATH left-to-right and uses the first match, an attacker who can write to any folder that appears **before** the real `python` in PATH can plant a malicious `python` binary there. When you type `python`, you run the attacker's version.

```bash
# Attacker places malicious 'python' in /tmp/evil/
# If /tmp/evil/ is somehow first in PATH:
PATH = /tmp/evil:/usr/local/bin:/usr/bin
python myScript.py   # runs /tmp/evil/python — the attacker's version
```

**Defence:** `echo $PATH` and `which python` before running anything sensitive. Never add world-writable directories to PATH.


In [8]:
# Security check: inspect PATH for suspicious entries
# A defensive script that flags risky PATH configurations

import os
import stat
from pathlib import Path

path_entries = os.environ.get("PATH", "").split(os.pathsep)

print("PATH Security Audit")
print("=" * 50)

for i, entry in enumerate(path_entries):
    p = Path(entry)
    issues = []

    if not p.exists():
        issues.append("does not exist")
    elif p.exists():
        try:
            mode = p.stat().st_mode
            # Check if world-writable (anyone can put files here)
            if mode & stat.S_IWOTH:
                issues.append("WORLD-WRITABLE -- PATH hijacking risk")
        except PermissionError:
            issues.append("permission denied")

    # Flag temp directories as suspicious if they appear early in PATH
    if any(tmp in str(p).lower() for tmp in ["/tmp", "/var/tmp", "temp"]):
        issues.append("temp directory in PATH")

    status = "[WARN] " + " | ".join(issues) if issues else "[OK]  "
    print(f"  [{i+1:2}] {status} {entry}")


PATH Security Audit
  [ 1] [WARN] WORLD-WRITABLE -- PATH hijacking risk c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv\Scripts
  [ 2] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv\Scripts
  [ 3] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Users\Zoltan\AppData\Local\Programs\Microsoft VS Code
  [ 4] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Windows\system32
  [ 5] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Windows
  [ 6] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Windows\System32\Wbem
  [ 7] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Windows\System32\WindowsPowerShell\v1.0\
  [ 8] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Windows\System32\OpenSSH\
  [ 9] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Program Files\PowerShell\7\
  [10] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Program Files\Git\cmd
  [11] [WARN] WORLD-WRITABLE -- PATH hijacking risk C:\Users\Zoltan\AppD

In [10]:
# Check if a package name looks like a typosquat of a known legitimate package
# A simple edit-distance check

def levenshtein(s1, s2):
    """Compute edit distance between two strings."""
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if not s2:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

known_packages = ["requests", "scapy", "paramiko", "flask", "django", "pandas"]
suspicious_name = "requets"   # change this to test different names

print(f"Checking '{suspicious_name}' against known packages:")
for pkg in known_packages:
    distance = levenshtein(suspicious_name.lower(), pkg.lower())
    if distance <= 2 and suspicious_name.lower() != pkg.lower():
        print(f"  [WARN] '{suspicious_name}' is {distance} edit(s) away from '{pkg}' -- possible typosquat")


Checking 'requets' against known packages:
  [WARN] 'requets' is 1 edit(s) away from 'requests' -- possible typosquat


### Practice - Section 5

**Write** — A junior analyst is about to install the following packages. Use the `levenshtein` function from above to check each one against the `known_packages` list. Flag anything within 2 edits of a known package.


In [14]:
# Packages the analyst wants to install
to_install = ["requests", "scappy", "parammiko", "flask", "djano"]
known_packages = ["requests", "scapy", "paramiko", "flask", "django", "pandas"]

# Check each one — flag any that look like typosquats of known packages
# Your code here
for ipkg in to_install:
    print(f"Checking '{ipkg}' against known packages:")
    for pkg in known_packages:
        distance = levenshtein(ipkg, pkg)
        if distance <= 2 and ipkg != pkg:
            print(f"  [WARN] '{ipkg}' is {distance} edit(s) away from '{pkg}' -- possible typosquat")



Checking 'requests' against known packages:
Checking 'scappy' against known packages:
  [WARN] 'scappy' is 1 edit(s) away from 'scapy' -- possible typosquat
Checking 'parammiko' against known packages:
  [WARN] 'parammiko' is 1 edit(s) away from 'paramiko' -- possible typosquat
Checking 'flask' against known packages:
Checking 'djano' against known packages:
  [WARN] 'djano' is 1 edit(s) away from 'django' -- possible typosquat


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
to_install    = ["requests", "scappy", "parammiko", "flask", "djano"]
known_packages = ["requests", "scapy", "paramiko", "flask", "django", "pandas"]

for name in to_install:
    if name in known_packages:
        print(f"[OK]   '{name}' matches a known package exactly")
        continue
    flagged = []
    for known in known_packages:
        if levenshtein(name.lower(), known.lower()) <= 2:
            flagged.append(known)
    if flagged:
        print(f"[WARN] '{name}' looks like a typosquat of: {', '.join(flagged)}")
    else:
        print(f"[OK]   '{name}' no close matches found")
```

</details>

---

## 6. Modern Alternatives: pipenv and poetry

`venv` + `pip` is the standard, built-in approach. Two modern tools extend this with better dependency resolution and locking. You do not need to use them — but you will encounter them in professional codebases.

### pipenv

Combines `venv` + `pip` into one tool. Introduces a `Pipfile` (readable dependency specification) and `Pipfile.lock` (exact pinned versions, like `requirements.txt` but more structured).

```bash
pip install pipenv
pipenv install requests           # creates venv + installs + updates Pipfile
pipenv install --dev pytest       # dev-only dependency
pipenv shell                      # activate the managed environment
pipenv lock                       # pin all versions to Pipfile.lock
```

### poetry

A more complete project management tool. Handles packaging, publishing to PyPI, and dependency resolution. Uses `pyproject.toml` as the configuration file (the modern Python standard).

```bash
pip install poetry
poetry new my_project             # scaffold a new project
poetry add requests               # add a dependency
poetry add --group dev pytest     # add dev dependency
poetry install                    # install all dependencies
poetry shell                      # activate the environment
```

### Which to use?

| Tool | Use when |
|------|----------|
| `venv` + `pip` | Learning, simple scripts, broad compatibility. **Start here.** |
| `pipenv` | Teams that want `Pipfile` + lock file without switching completely |
| `poetry` | Full project management, publishing packages, complex dependencies |

> For this course: stick with `venv` + `pip` + `requirements.txt`. Master the fundamentals before adopting wrapper tools.


---

## 7. Summary

| Concept | Key rule |
|---------|----------|
| **PATH** | An ordered list of folders the OS searches to find executables. First match wins. |
| **Activation** | Prepends `.venv/bin` to PATH so the venv Python and pip are found before the system ones. |
| **pyvenv.cfg** | Config file inside `.venv` — records which system Python the venv was created from. |
| **site-packages** | The folder inside `.venv` where all installed packages actually live. |
| **sys.path** | Python's import search list. The venv's site-packages appears early to override system packages. |
| **Typosquatting** | Malicious packages with names one keystroke from legitimate ones. Always verify on pypi.org. |
| **Dependency confusion** | Attacker publishes a higher-versioned package on PyPI with the same name as an internal private package. |
| **PATH hijacking** | Malicious binary planted in a folder that appears before the real one in PATH. Check `which python`. |
| **pipenv / poetry** | Modern alternatives to venv+pip. Useful eventually. Not needed at this stage. |

---

### Next Steps

You now understand virtual environments at the implementation level — not just how to use them but why they work.

**-> [02_Drills_VirtualEnv.ipynb](02_Drills_VirtualEnv.ipynb)** — environment inspection exercises that run inside a live virtual environment to verify your setup.
